# Similar Movies Debug Runner

Run the setup cell once, set `tmdb_ids`, then execute the result cell to inspect the standalone similar-movies flow with lane labels.

## Cell 1 - Setup

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from db.postgres import fetch_movie_cards, pool as postgres_pool
from search_v2.similar_movies import run_similar_movies_for_ids

if postgres_pool._closed:
    await postgres_pool.open()

print(f"Project root: {PROJECT_ROOT}")
print("Postgres pool open")

/Users/michaelkeohane/Documents/movie-finder-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root: /Users/michaelkeohane/Documents/movie-finder-rag
Postgres pool open


## Cell 2 - Anchors

## Cell 3 - Results

In [7]:
tmdb_ids = [17473]  # Inception
limit = 25

def year_from_release_ts(release_ts: int | None) -> int | None:
    if release_ts is None:
        return None
    return datetime.fromtimestamp(release_ts, tz=timezone.utc).year


result = await run_similar_movies_for_ids(tmdb_ids, limit=limit)

cards = await fetch_movie_cards([item.movie_id for item in result.ranked])
cards_by_id = {card["movie_id"]: card for card in cards}

rows = []
for rank, item in enumerate(result.ranked, start=1):
    card = cards_by_id.get(item.movie_id, {})
    lane_scores = item.evidence.lane_scores
    rows.append(
        {
            "rank": rank,
            "title": card.get("title", "<missing card>"),
            "year": year_from_release_ts(card.get("release_ts")),
            "tmdb_id": item.movie_id,
            "score": round(item.score, 4),
            "dominant_lane": item.evidence.dominant_lane,
            "lanes": ", ".join(item.evidence.candidate_sources),
            "shape": round(lane_scores.get("shape", 0.0), 4),
            "director": round(lane_scores.get("director", 0.0), 4),
            "franchise": round(lane_scores.get("franchise", 0.0), 4),
            "studio": round(lane_scores.get("studio", 0.0), 4),
            "source": round(lane_scores.get("source", 0.0), 4),
            "quality": round(lane_scores.get("quality", 0.0), 4),
        }
    )

print("anchors:", result.anchor_movie_ids)
print("active_anchor_types:", result.active_anchor_types)
print("lane_weights:", result.debug.normalized_lane_weights)
print("vector_space_weights:", result.debug.vector_space_weights)
display(pd.DataFrame(rows))

anchors: [17473]
active_anchor_types: ['standard_shape', 'cult_garbage']
lane_weights: {'shape': 0.4310344827586207, 'director': 0.10344827586206896, 'franchise': 0.10344827586206896, 'studio': 0.05172413793103448, 'source': 0.03448275862068966, 'quality': 0.2758620689655173}
vector_space_weights: {'anchor': 0.12264150943396228, 'plot_events': 0.0660377358490566, 'plot_analysis': 0.18867924528301888, 'viewer_experience': 0.18867924528301888, 'watch_context': 0.12264150943396228, 'narrative_techniques': 0.0660377358490566, 'production': 0.12264150943396228, 'reception': 0.12264150943396228}


,rank,title,year,tmdb_id,score,dominant_lane,lanes,shape,director,franchise,studio,source,quality
0,1,Troll 2,1990,26914,0.5252,shape,"shape, quality",0.6890,0.0,0.0,0.0,0.0,0.8272
1,2,Space Mutiny,1988,32148,0.4927,shape,"shape, quality",0.6539,0.0,0.0,0.0,0.0,0.7644
2,3,Betrayed: A Story of Three Women,1995,978193,0.4310,shape,shape,1.0000,0.0,0.0,0.0,0.0,0.0000
3,4,A Complete History of My Sexual Failures,2008,13206,0.4170,shape,shape,0.9675,0.0,0.0,0.0,0.0,0.0000
4,5,Movie 43,2013,87818,0.4106,quality,"shape, quality",0.4021,0.0,0.0,0.0,0.0,0.8600
5,6,Birdemic: Shock and Terror,2010,40016,0.3969,quality,"shape, quality",0.3016,0.0,0.0,0.0,0.0,0.9676
6,7,Manos: The Hands of Fate,1966,22293,0.3767,quality,"shape, quality",0.2349,0.0,0.0,0.0,0.0,0.9984
7,8,Birdemic 2: The Resurrection,2013,188489,0.3719,quality,"shape, quality",0.4190,0.0,0.0,0.0,0.0,0.6935
8,9,This Is Me…Now,2024,1217343,0.3382,shape,shape,0.7847,0.0,0.0,0.0,0.0,0.0000
9,10,After the Screaming Stops,2018,545577,0.3268,shape,shape,0.7581,0.0,0.0,0.0,0.0,0.0000
